<a href="https://colab.research.google.com/github/codematser69/candidate-ranking--dashboard/blob/main/panda_tech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#	============================================================
#	CELL	1:	Mount	Google	Drive
#	============================================================
from	google.colab	import	drive
drive.mount('/content/drive')
import	os
print(os.listdir('/content/drive/MyDrive/redrob_hackathon'))

Mounted at /content/drive
['candidates.jsonl', 'job_description.docx', 'sample_candidates.json', 'validate_submission.py', 'submission_spec.docx', 'submission_metadata_template.yaml', 'candidate_schema.json', 'redrob_signals_doc.docx', 'sample_submission.csv', 'output']


In [ ]:
#	============================================================
#	CELL	2:	Install	only	what's	missing	on	Colab
#	============================================================
!pip	install	-q	lightgbm	tqdm
print("Libraries	installed")

Libraries	installed


In [ ]:
#	============================================================
#	CELL	3:	Imports	&	Config
#	============================================================
import	json
import	os
import	re
import	math
import	time
import	warnings
from	datetime	import	datetime
import	numpy	as	np
import	pandas	as	pd
from	tqdm	import	tqdm
from	sklearn.preprocessing	import	MinMaxScaler
from	sklearn.feature_extraction.text	import	TfidfVectorizer
from	sklearn.metrics.pairwise	import	cosine_similarity
import	lightgbm	as	lgb
warnings.filterwarnings("ignore")
#	CONFIRMED	real	folder	name:	redrob_hackathon	(underscore,	not	hyphen)
DRIVE_BASE	=	"/content/drive/MyDrive/redrob_hackathon"
CANDIDATES_PATH	=	f"{DRIVE_BASE}/candidates.jsonl"
SAMPLE_PATH	=	f"{DRIVE_BASE}/sample_candidates.json"
OUTPUT_DIR	=	f"{DRIVE_BASE}/output"
os.makedirs(OUTPUT_DIR,	exist_ok=True)
REFERENCE_DATE	=	"2026-06-20"
#	"today"	for	recency	calculations
print("Config	ready")
print(f"Full	pool	path	:	{CANDIDATES_PATH}		|	exists:	{os.path.exists(CANDIDATES_PATH)}")
print(f"Sample	path					:	{SAMPLE_PATH}		|	exists:	{os.path.exists(SAMPLE_PATH)}")
print(f"Output	path					:	{OUTPUT_DIR}")

Config	ready
Full	pool	path	:	/content/drive/MyDrive/redrob_hackathon/candidates.jsonl		|	exists:	True
Sample	path					:	/content/drive/MyDrive/redrob_hackathon/sample_candidates.json		|	exists:	True
Output	path					:	/content/drive/MyDrive/redrob_hackathon/output


In [ ]:
# ============================================================
# CELL 4: Load all 100K candidates AND the 50 sample candidates
# ============================================================
def load_candidates(path):
    candidates = []
    with open(path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Loading candidates", unit=" lines"):
            line = line.strip()
            if line:
                candidates.append(json.loads(line))
    return candidates

start = time.time()
candidates_full = load_candidates(CANDIDATES_PATH)
print(f"Loaded {len(candidates_full):,} candidates from full pool in {time.time()-start:.1f}s")

with open(SAMPLE_PATH, "r") as f:
    candidates_sample = json.load(f)
print(f"Loaded {len(candidates_sample)} sample candidates (this is your TRAINING set)")

# Build a lookup so any candidate_id (from either set) can be retrieved later
candidates_lookup = {c["candidate_id"]: c for c in candidates_full}
for c in candidates_sample:
    candidates_lookup.setdefault(c["candidate_id"], c)


Loading candidates: 100000 lines [00:38, 2609.14 lines/s]


Loaded 100,000 candidates from full pool in 38.4s
Loaded 50 sample candidates (this is your TRAINING set)


In [ ]:
# ============================================================
# CELL 5: Shared helper functions (used by both datasets)
# ============================================================
REFERENCE_DATE = "2026-06-20" # Added for self-contained execution
def days_since(date_str, reference_date=REFERENCE_DATE):
    if not date_str:
        return 999
    try:
        d = datetime.strptime(date_str, "%Y-%m-%d")
        ref = datetime.strptime(reference_date, "%Y-%m-%d")
        return (ref - d).days
    except Exception:
        return 999

def get_best_education_tier(education_list):
    if not education_list:
        return 3
    tiers = []
    for edu in education_list:
        t = edu.get("tier", "")
        match = re.search(r"(\d+)", str(t))
        if match:
            tiers.append(int(match.group(1)))
    return min(tiers) if tiers else 3

def build_feature_row(c):
    """Flatten ONE candidate record. Shared by both the 100K pool and
    the 50-sample set so flattening logic never drifts between them."""
    cid = c.get("candidate_id", "")
    profile = c.get("profile", {}) or {}
    signals = c.get("redrob_signals", {}) or {}
    career_history = c.get("career_history", []) or []
    education = c.get("education", []) or []
    skills = c.get("skills", []) or []

    row = {
        "candidate_id": cid,
        "current_title": profile.get("current_title", ""),
        "current_company": profile.get("current_company", ""),
        "current_industry": profile.get("current_industry", ""),
        "current_company_size": profile.get("current_company_size", ""),
        "location_city": profile.get("location", ""),
        "country": profile.get("country", ""),
        "total_exp_years": profile.get("years_of_experience", 0) or 0,
        "headline": profile.get("headline", ""),
        "summary": profile.get("summary", ""),
        "notice_period_days": signals.get("notice_period_days", 999) or 999,
        "willing_to_relocate": int(bool(signals.get("willing_to_relocate", False))),
        "open_to_work_flag": int(bool(signals.get("open_to_work_flag", False))),
        "last_active_days_ago": days_since(signals.get("last_active_date", "")),
        "recruiter_response_rate": signals.get("recruiter_response_rate", 0) or 0,
        "github_activity_score": signals.get("github_activity_score", 0) or 0,
        "profile_completeness_score": signals.get("profile_completeness_score", 0) or 0,
        "interview_completion_rate": signals.get("interview_completion_rate", 0) or 0,
        "offer_acceptance_rate": signals.get("offer_acceptance_rate", 0) or 0,
        "education_tier": get_best_education_tier(education),
        "skills_raw": json.dumps(skills, default=str),
        "career_history_raw": json.dumps(career_history, default=str),
        "education_raw": json.dumps(education, default=str),
        "full_text": "",
    }

    for sig_key, sig_val in signals.items():
        if isinstance(sig_val, (int, float, bool)):
            row[f"sig_{sig_key}"] = float(sig_val)

    skills_list = [s.get("name", "") for s in skills if isinstance(s, dict)]
    career_text = []
    for job in career_history:
        if isinstance(job, dict):
            career_text.append(job.get("title", ""))
            career_text.append(job.get("description", ""))
    education_text = []
    for edu in education:
        if isinstance(edu, dict):
            education_text.append(edu.get("degree", ""))
            education_text.append(edu.get("field_of_study", ""))
            education_text.append(edu.get("institution", ""))

    row["full_text"] = " ".join(filter(None, [
        row["current_title"], row["headline"], " ".join(skills_list),
        " ".join(career_text), " ".join(education_text), row["summary"],
    ])).lower()

    return row

def flatten_to_dataframe(candidates, desc="Flattening"):
    rows = [build_feature_row(c) for c in tqdm(candidates, desc=desc)]
    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# CELL 6: Flatten both datasets
# ============================================================
df_full = flatten_to_dataframe(candidates_full, desc="Flattening 100K pool")
df_sample = flatten_to_dataframe(candidates_sample, desc="Flattening 50 sample")

print(f"Full pool shape  : {df_full.shape}")
print(f"Sample set shape : {df_sample.shape}")

Flattening 50 sample: 100%|██████████| 50/50 [00:00<00:00, 2315.25it/s]

Full pool shape  : (100000, 42)
Sample set shape : (50, 42)


In [ ]:
# ============================================================
# CELL 7: Shared cleaning function
# ============================================================
def normalize_text(x):
    if not isinstance(x, str):
        return ""
    return " ".join(x.strip().lower().split())

def clean_dataframe(df, label="dataset"):
    print(f"\n=== CLEANING: {label} ===")

    key_fields = ["current_title", "current_company", "location_city", "country", "summary"]
    for field in key_fields:
        missing_pct = (df[field].isna() | (df[field].astype(str).str.strip() == "")).mean() * 100
        print(f"  {field:20s}: {missing_pct:5.1f}% missing/empty")

    df["has_no_career_history"] = df["career_history_raw"].apply(lambda x: json.loads(x) == [])
    df["has_no_education"] = df["education_raw"].apply(lambda x: json.loads(x) == [])
    df["has_no_skills"] = df["skills_raw"].apply(lambda x: json.loads(x) == [])

    df["total_exp_years"] = df["total_exp_years"].clip(lower=0, upper=50)
    df["notice_period_days"] = df["notice_period_days"].clip(lower=0, upper=365)
    df["profile_completeness_score"] = df["profile_completeness_score"].clip(lower=0, upper=100)
    df["recruiter_response_rate"] = df["recruiter_response_rate"].clip(lower=0, upper=1)
    df["interview_completion_rate"] = df["interview_completion_rate"].clip(lower=0, upper=1)
    df["offer_acceptance_rate"] = df["offer_acceptance_rate"].clip(lower=0, upper=1)

    dup_count = df["candidate_id"].duplicated().sum()
    if dup_count > 0:
        print(f"  Dropping {dup_count} duplicate candidate_ids")
        df = df.drop_duplicates(subset="candidate_id", keep="first").reset_index(drop=True)

    df["current_company_clean"] = df["current_company"].apply(normalize_text)
    df["current_industry_clean"] = df["current_industry"].apply(normalize_text)
    df["location_city_clean"] = df["location_city"].apply(normalize_text)
    df["country_clean"] = df["country"].apply(normalize_text)
    df["current_title_clean"] = df["current_title"].apply(normalize_text)

    print(f"  Final row count: {len(df):,}")
    return df

In [ ]:
# ============================================================
# CELL 8: Clean both datasets
# ============================================================
df_full = clean_dataframe(df_full, label="full 100K pool")
df_sample = clean_dataframe(df_sample, label="50 sample (training set)")


=== CLEANING: full 100K pool ===
  current_title       :   0.0% missing/empty
  current_company     :   0.0% missing/empty
  location_city       :   0.0% missing/empty
  country             :   0.0% missing/empty
  summary             :   0.0% missing/empty
  Final row count: 100,000

=== CLEANING: 50 sample (training set) ===
  current_title       :   0.0% missing/empty
  current_company     :   0.0% missing/empty
  location_city       :   0.0% missing/empty
  country             :   0.0% missing/empty
  summary             :   0.0% missing/empty
  Final row count: 50


In [ ]:
# ============================================================
# CELL 9: Checkpoint the cleaned full pool (skip re-running 6-8 next session)
# ============================================================
df_full.to_parquet(f"{OUTPUT_DIR}/candidates_full_clean.parquet", index=False)
df_sample.to_parquet(f"{OUTPUT_DIR}/candidates_sample_clean.parquet", index=False)
print("Checkpoints saved.")

# To reload in a future session instead of re-running Cells 4-8:
# df_full = pd.read_parquet(f"{OUTPUT_DIR}/candidates_full_clean.parquet")
# df_sample = pd.read_parquet(f"{OUTPUT_DIR}/candidates_sample_clean.parquet")

Checkpoints saved.


In [ ]:
# ============================================================
# CELL 10: Paste the FULL job_description text here
# ============================================================
JOB_DESCRIPTION = """
[PASTE THE ENTIRE job_description.docx TEXT HERE — every section,
 from the title down to "Good luck." Paste the full narrative, not
 just the skills list — TF-IDF benefits from the surrounding context.]
"""

JD_REQUIRED_SKILLS = [
    "embeddings", "embedding", "sentence-transformers", "bge", "e5",
    "vector database", "pinecone", "weaviate", "qdrant", "milvus",
    "opensearch", "elasticsearch", "faiss", "hybrid search",
    "python", "ndcg", "mrr", "map", "a/b test", "ranking", "retrieval",
]
JD_NICE_SKILLS = [
    "lora", "qlora", "peft", "fine-tuning", "learning-to-rank", "xgboost",
    "lightgbm", "hr-tech", "recruiting", "distributed systems",
    "inference optimization", "open source", "github",
]
JD_DISQUALIFYING_COMPANIES = ["tcs", "infosys", "wipro", "accenture", "cognizant", "capgemini"]
JD_PREFERRED_LOCATIONS = ["pune", "noida", "hyderabad", "mumbai", "delhi", "ncr"]
JD_EXP_MIN = 5
JD_EXP_MAX = 9
JD_NOTICE_IDEAL = 30

print(f"JD loaded | Required skills: {len(JD_REQUIRED_SKILLS)} | Nice: {len(JD_NICE_SKILLS)}")

JD loaded | Required skills: 21 | Nice: 13


In [ ]:
# ============================================================
# CELL 11: Scoring functions (used identically on both datasets)
# ============================================================
def compute_keyword_score(text, required, nice):
    text = text.lower()
    req_hits = sum(1 for s in required if s in text)
    nice_hits = sum(1 for s in nice if s in text)
    return (req_hits / max(len(required), 1), nice_hits / max(len(nice), 1), req_hits)

def exp_fit(years, ideal_min, ideal_max):
    if years < 0:
        return 0.0
    if ideal_min <= years <= ideal_max:
        return 1.0
    gap = ideal_min - years if years < ideal_min else years - ideal_max
    return math.exp(-0.3 * gap)

def recency_score(days_ago):
    return math.exp(-max(float(days_ago), 0) / 180)

def notice_score(days, ideal_max=JD_NOTICE_IDEAL):
    days = max(float(days), 0)
    if days <= ideal_max:
        return 1.0
    return math.exp(-0.02 * (days - ideal_max))

def location_score(city, country, preferred_locs, willing_to_relocate):
    city_l = str(city).lower()
    country_l = str(country).lower()
    if any(loc in city_l for loc in preferred_locs):
        return 1.0
    if country_l == "india":
        return 0.7
    return 0.4 if willing_to_relocate else 0.1

def edu_tier_score(tier):
    tier = max(1.0, min(5.0, float(tier)))
    return (6.0 - tier) / 5.0

def it_services_penalty(career_history_raw, current_industry_clean):
    try:
        history = json.loads(career_history_raw)
    except Exception:
        history = []
    if not history:
        return 0.5 if current_industry_clean == "it services" else 1.0
    services_count = sum(
        1 for job in history
        if isinstance(job, dict) and any(
            comp in str(job.get("company", "")).lower() or
            str(job.get("industry", "")).lower() == "it services"
            for comp in JD_DISQUALIFYING_COMPANIES
        )
    )
    ratio = services_count / max(len(history), 1)
    if ratio >= 0.99:
        return 0.3
    elif ratio >= 0.5:
        return 0.7
    return 1.0

def detect_honeypot(c, total_exp_years, current_year=2026):
    flags = 0
    career_history = c.get("career_history", []) or []
    education = c.get("education", []) or []
    skills = c.get("skills", []) or []

    for job in career_history:
        if isinstance(job, dict) and (job.get("duration_months", 0) or 0) > 15 * 12:
            flags += 1

    expert_zero = sum(
        1 for sk in skills
        if isinstance(sk, dict)
        and str(sk.get("proficiency", "")).lower() in ("expert", "advanced")
        and (sk.get("duration_months", 1) or 1) == 0
    )
    if expert_zero >= 3:
        flags += 2

    for edu in education:
        if isinstance(edu, dict) and edu.get("end_year"):
            max_possible = current_year - int(edu["end_year"])
            if total_exp_years > max_possible + 2:
                flags += 3

    if len(skills) > 40:
        flags += 1

    return min(flags / 5.0, 1.0)

In [ ]:
# ============================================================
# CELL 12: engineer_features() — applied identically to both datasets
# ============================================================
def engineer_features(df, candidates_lookup):
    corpus = df["full_text"].fillna("").tolist()
    jd_text = JOB_DESCRIPTION.lower()

    tfidf = TfidfVectorizer(
        max_features=50_000, ngram_range=(1, 2),
        min_df=1 if len(df) < 1000 else 3,   # smaller min_df for the tiny 50-row set
        sublinear_tf=True, strip_accents="unicode", analyzer="word", stop_words="english",
    )
    tfidf.fit(corpus + [jd_text])
    candidate_matrix = tfidf.transform(corpus)
    jd_vector = tfidf.transform([jd_text])
    df["score_tfidf"] = cosine_similarity(jd_vector, candidate_matrix).flatten()

    kw_results = df["full_text"].apply(lambda t: compute_keyword_score(t, JD_REQUIRED_SKILLS, JD_NICE_SKILLS))
    df["score_kw_required"] = kw_results.apply(lambda x: x[0])
    df["score_kw_nice"] = kw_results.apply(lambda x: x[1])

    df["score_exp_fit"] = df["total_exp_years"].apply(lambda y: exp_fit(float(y), JD_EXP_MIN, JD_EXP_MAX))
    df["score_recency"] = df["last_active_days_ago"].apply(recency_score)
    df["score_notice"] = df["notice_period_days"].apply(notice_score)
    df["score_location"] = df.apply(
        lambda r: location_score(r["location_city_clean"], r["country_clean"],
                                  JD_PREFERRED_LOCATIONS, bool(r["willing_to_relocate"])),
        axis=1
    )
    df["score_edu"] = df["education_tier"].apply(edu_tier_score)
    df["score_it_services_fit"] = df.apply(
        lambda r: it_services_penalty(r["career_history_raw"], r["current_industry_clean"]), axis=1
    )

    sig_cols = [c for c in df.columns if c.startswith("sig_")]
    if sig_cols:
        df[sig_cols] = df[sig_cols].fillna(0)
        scaler = MinMaxScaler()
        norm = pd.DataFrame(scaler.fit_transform(df[sig_cols]),
                             columns=[f"norm_{c}" for c in sig_cols], index=df.index)
        df = pd.concat([df, norm], axis=1)
        df["score_signals"] = df[[f"norm_{c}" for c in sig_cols]].mean(axis=1)
    else:
        df["score_signals"] = 0.5

    honeypot_scores = []
    for cid, exp_years in zip(df["candidate_id"], df["total_exp_years"]):
        c = candidates_lookup.get(cid, {})
        honeypot_scores.append(detect_honeypot(c, exp_years))
    df["honeypot_score"] = honeypot_scores

    return df

In [ ]:
# ============================================================
# CELL 13: Run feature engineering on BOTH datasets
# ============================================================
df_full = engineer_features(df_full, candidates_lookup)
df_sample = engineer_features(df_sample, candidates_lookup)

print("Full pool features computed:", df_full.shape)
print("Sample features computed   :", df_sample.shape)

Full pool features computed: (100000, 79)
Sample features computed   : (50, 79)


In [ ]:
# ============================================================
# CELL 14: Weighted composite score — computed on BOTH datasets
# ============================================================
WEIGHTS = {
    "score_tfidf": 0.25, "score_kw_required": 0.15, "score_exp_fit": 0.12,
    "score_signals": 0.15, "score_it_services_fit": 0.10, "score_recency": 0.08,
    "score_kw_nice": 0.05, "score_notice": 0.05, "score_location": 0.04, "score_edu": 0.01,
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 0.001, "Weights must sum to 1.0"

def compute_composite_score(df):
    df["composite_score"] = sum(df[feat] * w for feat, w in WEIGHTS.items())
    df["composite_score"] = df["composite_score"] * (1 - df["honeypot_score"] * 0.9)
    return df

df_full = compute_composite_score(df_full)
df_sample = compute_composite_score(df_sample)

print(df_full[["candidate_id", "composite_score"]].describe())

       composite_score
count    100000.000000
mean          0.287459
std           0.088682
min           0.017214
25%           0.249563
50%           0.308614
75%           0.350013
max           0.575124


In [ ]:
# ============================================================
# CELL 15: Print all 50 candidates one by one for manual labeling
# ============================================================
for c in candidates_sample:
    print("="*80)
    print(c["candidate_id"])
    print("Title:", c["profile"].get("current_title"), "@", c["profile"].get("current_company"))
    print("Industry:", c["profile"].get("current_industry"))
    print("Location:", c["profile"].get("location"), c["profile"].get("country"))
    print("Experience:", c["profile"].get("years_of_experience"), "years")
    print("Summary:", c["profile"].get("summary"))
    print("Skills:", [s.get("name") for s in c.get("skills", [])])

CAND_0000001
Title: Backend Engineer @ Mindtree
Industry: IT Services
Location: Toronto Canada
Experience: 6.9 years
Summary: Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.
Skills: ['Tailwind', 'NLP', 'Image Classification', 'Fine-tuning LLMs', 'Weights & Biases', 'Speech Recognition', 'Photoshop', 'TTS', 'LoRA', 'Apache Beam', 'AWS', 'Flask', 'BentoML', 'Milvus', 'GANs', 'Statistical Modeling', 'GCP']
CAND_0

In [ ]:
# ============================================================
# CELL 16: YOUR MANUAL LABELS — converted from candidate_ranking_top50
# 0 = not fit, 1 = weak, 2 = moderate, 3 = good fit, 4 = excellent fit
# ============================================================
MANUAL_LABELS = {
    "CAND_0000031": 4,  # rank 1, 92/100, strong fit
    "CAND_0000043": 4,  # rank 2, 74/100, strong fit
    "CAND_0000014": 4,  # rank 3, 68/100, strong fit
    "CAND_0000038": 3,  # rank 4, 63/100, good fit
    "CAND_0000027": 3,  # rank 5, 58/100, good fit
    "CAND_0000015": 3,  # rank 6, 55/100, good fit
    "CAND_0000032": 3,  # rank 7, 52/100, good fit
    "CAND_0000001": 3,  # rank 8, 50/100, good fit
    "CAND_0000010": 3,  # rank 9, 48/100, good fit
    "CAND_0000011": 3,  # rank 10, 45/100, good fit
    "CAND_0000021": 2,  # rank 11, 43/100, moderate fit
    "CAND_0000025": 2,  # rank 12, 38/100, moderate fit
    "CAND_0000035": 2,  # rank 13, 35/100, moderate fit
    "CAND_0000026": 2,  # rank 14, 33/100, moderate fit
    "CAND_0000019": 2,  # rank 15, 31/100, moderate fit
    "CAND_0000048": 2,  # rank 16, 29/100, moderate fit
    "CAND_0000008": 2,  # rank 17, 27/100, moderate fit
    "CAND_0000020": 1,  # rank 18, 22/100, weak fit
    "CAND_0000044": 1,  # rank 19, 20/100, weak fit
    "CAND_0000050": 1,  # rank 20, 18/100, weak fit
    "CAND_0000041": 1,  # rank 21, 16/100, weak fit
    "CAND_0000018": 1,  # rank 22, 14/100, weak fit
    "CAND_0000045": 1,  # rank 23, 13/100, weak fit
    "CAND_0000036": 1,  # rank 24, 12/100, weak fit
    "CAND_0000005": 1,  # rank 25, 11/100, weak fit
    "CAND_0000013": 1,  # rank 26, 10/100, weak fit
    "CAND_0000033": 0,  # rank 27, 9/100, disqualified
    "CAND_0000042": 0,  # rank 28, 8/100, disqualified
    "CAND_0000007": 0,  # rank 29, 7/100, disqualified
    "CAND_0000023": 0,  # rank 30, 7/100, disqualified
    "CAND_0000006": 0,  # rank 31, 5/100, disqualified
    "CAND_0000017": 0,  # rank 32, 5/100, disqualified
    "CAND_0000029": 0,  # rank 33, 4/100, disqualified
    "CAND_0000024": 0,  # rank 34, 4/100, disqualified
    "CAND_0000002": 0,  # rank 35, 4/100, disqualified
    "CAND_0000016": 0,  # rank 36, 3/100, disqualified
    "CAND_0000012": 0,  # rank 37, 3/100, disqualified
    "CAND_0000028": 0,  # rank 38, 3/100, disqualified
    "CAND_0000039": 0,  # rank 39, 3/100, disqualified
    "CAND_0000030": 0,  # rank 40, 3/100, disqualified
    "CAND_0000003": 0,  # rank 41, 2/100, disqualified
    "CAND_0000004": 0,  # rank 42, 2/100, disqualified
    "CAND_0000040": 0,  # rank 43, 2/100, disqualified
    "CAND_0000022": 0,  # rank 44, 2/100, disqualified
    "CAND_0000047": 0,  # rank 45, 2/100, disqualified
    "CAND_0000046": 0,  # rank 46, 1/100, disqualified
    "CAND_0000034": 0,  # rank 47, 1/100, disqualified
    "CAND_0000049": 0,  # rank 48, 1/100, disqualified
    "CAND_0000009": 0,  # rank 49, 1/100, disqualified
    "CAND_0000037": 0,  # rank 50, 1/100, disqualified
}

print(f"Labeled {len(MANUAL_LABELS)} out of 50 candidates")

Labeled 50 out of 50 candidates


In [ ]:
# ============================================================
# CELL 17: Train LightGBM on the hand-labeled 50-sample set
# ============================================================
FEATURE_NAMES = [
    "score_tfidf", "score_kw_required", "score_kw_nice", "score_exp_fit",
    "score_signals", "score_it_services_fit", "score_recency",
    "score_notice", "score_location", "score_edu",
    "total_exp_years", "education_tier",
]

df_sample_labeled = df_sample.copy()
df_sample_labeled["relevance_label"] = df_sample_labeled["candidate_id"].map(MANUAL_LABELS)
df_sample_labeled = df_sample_labeled.dropna(subset=["relevance_label"])

print(f"Training on {len(df_sample_labeled)} labeled examples")

if len(df_sample_labeled) < 10:
    print("⚠️  Too few labels — go back to Cell 16 and finish labeling before continuing.")
else:
    X_train = df_sample_labeled[FEATURE_NAMES].fillna(0)
    y_train = df_sample_labeled["relevance_label"].astype(int)
    group_train = [len(df_sample_labeled)]

    train_data = lgb.Dataset(X_train, label=y_train, group=group_train)
    params = {
        "objective": "lambdarank",
        "metric": "ndcg",
        "ndcg_eval_at": [10],
        "learning_rate": 0.05,
        "num_leaves": 7,        # small — guards against overfitting on ~50 rows
        "min_data_in_leaf": 3,
        "verbose": -1,
    }
    model = lgb.train(params, train_data, num_boost_round=50)
    print("Model trained on 50 labeled samples.")

Training on 50 labeled examples
Model trained on 50 labeled samples.


In [ ]:
# ============================================================
# CELL 18: Apply the trained model to ALL 100,000 candidates
# ============================================================
X_full = df_full[FEATURE_NAMES].fillna(0)
df_full["model_score"] = model.predict(X_full)

print("Model applied to full 100K pool.")
print(df_full["model_score"].describe())

Model applied to full 100K pool.
count    100000.000000
mean         -1.423259
std           1.342877
min          -2.802314
25%          -2.404149
50%          -1.907567
75%          -0.746146
max           3.027356
Name: model_score, dtype: float64
